In [0]:
%pip install azure-eventhub
dbutils.library.restartPython()

In [0]:
import json, random, time, uuid
from datetime import datetime, timezone, timedelta
from azure.eventhub import EventHubProducerClient, EventData

EVENTS_PER_SECOND = 5
RUN_SECONDS = 60          # keep short while testing to save credit
BAD_RATE = 0.05

conn = dbutils.secrets.get("kv-scope", "eh-send-conn")
kwargs = {} if "EntityPath=" in conn else {"eventhub_name": "orders"}
producer = EventHubProducerClient.from_connection_string(conn, **kwargs)

AREAS = {
    "Gachibowli": (17.4401, 78.3489), "Hitech City": (17.4435, 78.3772),
    "Madhapur": (17.4483, 78.3915), "Kondapur": (17.4600, 78.3640),
    "Banjara Hills": (17.4126, 78.4482), "Secunderabad": (17.4399, 78.4983),
}
STATUSES = ["placed", "accepted", "picked_up", "delivered"]
active = {}  # order_id -> state

def now(): return datetime.now(timezone.utc)

def new_order():
    area = random.choice(list(AREAS))
    oid = str(uuid.uuid4())
    active[oid] = {"area": area, "step": 0, "amount": round(random.uniform(150, 1200), 2)}
    return oid

def make_event():
    if not active or random.random() < 0.4:
        oid = new_order()
    else:
        oid = random.choice(list(active))
    s = active[oid]
    lat, lon = AREAS[s["area"]]
    if random.random() < 0.3:
        etype = "rider_ping"
    else:
        etype = "order_status"
        s["step"] = min(s["step"] + (0 if s["step"] == 0 and random.random() < 0.5 else 1), 3)
    ev = {
        "event_id": str(uuid.uuid4()), "event_type": etype, "order_id": oid,
        "customer_id": f"C{random.randint(1, 500)}", "area": s["area"],
        "amount": s["amount"], "status": STATUSES[s["step"]],
        "lat": round(lat + random.uniform(-0.01, 0.01), 5),
        "lon": round(lon + random.uniform(-0.01, 0.01), 5),
        "event_time": now().isoformat(),
    }
    if ev["status"] == "delivered": active.pop(oid, None)
    return ev

def corrupt(ev):
    kind = random.choice(["null_id", "late", "negative"])
    if kind == "null_id": ev["order_id"] = None
    elif kind == "late": ev["event_time"] = (now() - timedelta(minutes=10)).isoformat()
    else: ev["amount"] = -abs(ev["amount"])
    return ev

In [0]:
sent, last = 0, None
end = time.time() + RUN_SECONDS
try:
    while time.time() < end:
        batch = producer.create_batch()
        for _ in range(EVENTS_PER_SECOND):
            ev = make_event()
            if last and random.random() < BAD_RATE / 2:
                ev = dict(last)                       # duplicate
            elif random.random() < BAD_RATE / 2:
                ev = corrupt(ev)
            last = ev
            batch.add(EventData(json.dumps(ev)))
        producer.send_batch(batch)
        sent += EVENTS_PER_SECOND
        time.sleep(1)
    print(f"Sent {sent} events")
finally:
    producer.close()